# 05 Fill Additive Marking — local_3x3 H=4 bottleneck CNN, fast unique-patch training

Version 19 is a faster replacement for Version 18.

It keeps the submitted ONNX architecture `Conv3x3 10→4 + ReLU + Conv1x1 4→10`, but trains on the compressed table of unique local 3×3 patches instead of full 30×30 grids.

The notebook still refuses to create `submission.zip` unless all 8 tasks save models and every exported ONNX model has exact visible validation.

Export note: `torch.onnx.export(..., dynamo=False)` is used to force the legacy ONNX exporter and avoid requiring `onnxscript` on Kaggle.


In [ ]:
import json, math, shutil, subprocess, sys, time, zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

BATCH, CH, GRID_H, GRID_W = 1, 10, 30, 30
PATCH_DIM = CH * 3 * 3
FAMILY = 'fill_enclosed_regions'
MODEL_VERSION = 'fill-additive-local3x3-bottleneck-h4-fast-v0.19'
HIDDEN = 4

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
    except Exception:
        print('Installing', pip_name)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

ensure_package('onnx')
ensure_package('onnxruntime')
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort

def default_paths():
    kaggle_dir = Path('/kaggle/input/competitions/neurogolf-2026')
    if kaggle_dir.exists():
        root, data_dir = Path('/kaggle/working'), kaggle_dir
    else:
        root = Path.cwd()
        data_dir = root / 'competition_material' / 'taskfiles'
        if not data_dir.exists():
            data_dir = root / 'competition_material'
    out_dir = root / 'working_submission' / FAMILY
    out_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, out_dir

def load_task_type_map():
    candidates = [
        Path('/kaggle/input/datasets/prince22466/task-type-map-csv/task_type_map.csv'),
        Path('task_groups/task_type_map.csv'),
        Path('Co_Kaggle/g3/task_groups/task_type_map.csv'),
    ]
    for p in candidates:
        if p.exists():
            return pd.read_csv(p, dtype={'task_id': str})
    raise FileNotFoundError(f'task_type_map.csv not found in {candidates}')

def task_path(data_dir, task_id):
    name = f'{task_id}.json' if str(task_id).startswith('task') else f'task{int(task_id):03d}.json'
    for p in [Path(data_dir) / name, Path(data_dir) / 'taskfiles' / name]:
        if p.exists():
            return p
    raise FileNotFoundError(name)

def load_task(data_dir, task_id):
    with task_path(data_dir, task_id).open('r', encoding='utf-8') as f:
        return json.load(f)

def all_examples(task):
    return task.get('train', []) + task.get('test', []) + task.get('arc-gen', [])

def grid_to_chw(grid):
    arr = np.zeros((CH, GRID_H, GRID_W), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if 0 <= r < GRID_H and 0 <= c < GRID_W and 0 <= int(color) < CH:
                arr[int(color), r, c] = 1.0
    return arr

def grid_to_tensor(grid):
    return grid_to_chw(grid)[None].astype(np.float32)

def target_vec_at(output_grid, r, c):
    y = np.zeros((CH,), dtype=np.float32)
    if r < len(output_grid) and len(output_grid) > 0 and c < len(output_grid[0]):
        color = int(output_grid[r][c])
        if 0 <= color < CH:
            y[color] = 1.0
    return y

def patch_at_chw(x, r, c):
    p = np.zeros((CH, 3, 3), dtype=np.float32)
    for dy in range(-1, 2):
        rr = r + dy
        if rr < 0 or rr >= GRID_H:
            continue
        for dx in range(-1, 2):
            cc = c + dx
            if 0 <= cc < GRID_W:
                p[:, dy + 1, dx + 1] = x[:, rr, cc]
    return p

def build_unique_patch_dataset(task):
    mapping, conflicts, total = {}, [], 0
    for ex_i, ex in enumerate(all_examples(task)):
        x = grid_to_chw(ex['input'])
        for r in range(GRID_H):
            for c in range(GRID_W):
                patch = patch_at_chw(x, r, c)
                y = target_vec_at(ex['output'], r, c)
                key = patch.tobytes()
                total += 1
                if key not in mapping:
                    mapping[key] = (patch, y)
                elif mapping[key][1].tobytes() != y.tobytes():
                    conflicts.append((ex_i, r, c))
    xs = np.stack([v[0] for v in mapping.values()]).astype(np.float32)
    ys = np.stack([v[1] for v in mapping.values()]).astype(np.float32)
    return xs, ys, {'unique_patches': len(xs), 'total_positions': total, 'conflict_count': len(conflicts), 'conflicts': conflicts[:10]}

class PatchMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(PATCH_DIM, HIDDEN)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(HIDDEN, CH)
    def forward(self, patch):
        return self.fc2(self.relu(self.fc1(patch.reshape(patch.shape[0], -1))))

class ConvModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(CH, HIDDEN, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(HIDDEN, CH, kernel_size=1)
    def forward(self, x):
        return self.conv2(self.relu(self.conv1(x)))

def copy_mlp_to_conv(mlp):
    conv = ConvModel()
    with torch.no_grad():
        conv.conv1.weight.copy_(mlp.fc1.weight.reshape(HIDDEN, CH, 3, 3))
        conv.conv1.bias.copy_(mlp.fc1.bias)
        conv.conv2.weight.copy_(mlp.fc2.weight.reshape(CH, HIDDEN, 1, 1))
        conv.conv2.bias.copy_(mlp.fc2.bias)
    conv.eval()
    return conv

def exact_stats(model, x, y):
    with torch.no_grad():
        pred = (model(x) > 0).to(y.dtype)
        ok = (pred == y).all(dim=1)
    right = int(ok.sum().item())
    return right, int(y.shape[0]) - right, int(y.shape[0])

def train_fast(task, task_id, seeds=(0,1,2,3), max_epochs=1200, check_every=20, lr=0.05):
    x_np, y_np, info = build_unique_patch_dataset(task)
    if info['conflict_count']:
        return None, {'ok': False, 'task_id': task_id, 'reason': 'local patch conflicts', **info}
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    x = torch.from_numpy(x_np).to(device)
    y = torch.from_numpy(y_np).to(device)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.full((CH,), 3.0, device=device))
    best_wrong, best_loss, best_state, attempts = 10**9, float('inf'), None, []
    for seed in seeds:
        torch.manual_seed(seed); np.random.seed(seed)
        m = PatchMLP().to(device)
        opt = torch.optim.AdamW(m.parameters(), lr=lr)
        start = time.time()
        seed_best = {'seed': seed, 'epoch': None, 'wrong': 10**9, 'loss': float('inf')}
        for epoch in range(1, max_epochs + 1):
            opt.zero_grad(set_to_none=True)
            logits = m(x)
            bce = crit(logits, y)
            margin = (torch.relu(1.5 - logits) * y + torch.relu(1.5 + logits) * (1.0 - y)).mean()
            loss = bce + 0.10 * margin
            loss.backward(); opt.step()
            if epoch == 1 or epoch % check_every == 0 or epoch == max_epochs:
                right, wrong, total = exact_stats(m, x, y)
                lv = float(loss.item())
                if wrong < seed_best['wrong'] or (wrong == seed_best['wrong'] and lv < seed_best['loss']):
                    seed_best = {'seed': seed, 'epoch': epoch, 'right': right, 'wrong': wrong, 'total': total, 'loss': lv, 'elapsed_sec': time.time() - start}
                if wrong < best_wrong or (wrong == best_wrong and lv < best_loss):
                    best_wrong, best_loss = wrong, lv
                    best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
                if wrong == 0:
                    attempts.append(seed_best)
                    cpu_m = PatchMLP(); cpu_m.load_state_dict({k: v.cpu() for k, v in m.state_dict().items()}); cpu_m.eval()
                    return copy_mlp_to_conv(cpu_m), {'ok': True, 'task_id': task_id, 'trainer': 'h4_unique_patch_cnn', 'model_version': MODEL_VERSION, 'seed': seed, 'epoch': epoch, 'visible_patch_right': right, 'visible_patch_wrong': wrong, 'visible_patch_total': total, 'loss': lv, 'attempts': attempts, **info}
        attempts.append(seed_best)
        print(f"{task_id} seed={seed}: unique={info['unique_patches']} best_wrong={seed_best['wrong']} epoch={seed_best['epoch']} elapsed={seed_best.get('elapsed_sec',0):.1f}s")
    if best_state is not None:
        cpu_m = PatchMLP(); cpu_m.load_state_dict(best_state); cpu_m.eval()
        best_conv = copy_mlp_to_conv(cpu_m)
    else:
        best_conv = None
    return best_conv, {'ok': False, 'task_id': task_id, 'trainer': 'h4_unique_patch_cnn', 'model_version': MODEL_VERSION, 'reason': 'H=4 did not reach exact unique-patch match', 'best_visible_patch_wrong': best_wrong, 'best_loss': best_loss, 'attempts': attempts, **info}

def export_onnx(model, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    model.eval()
    dummy = torch.zeros((BATCH, CH, GRID_H, GRID_W), dtype=torch.float32)
    with torch.no_grad():
        torch.onnx.export(model, dummy, str(path), input_names=['input'], output_names=['output'], opset_version=10, do_constant_folding=True, dynamo=False)
    onnx.checker.check_model(str(path))
    return path

def run_onnx(path, input_grid):
    sess = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    out = sess.run(['output'], {'input': grid_to_tensor(input_grid)})[0]
    return (out > 0).astype(np.float32)

def validate_onnx(path, task):
    right = wrong = 0
    for ex in all_examples(task):
        expected = grid_to_tensor(ex['output'])
        actual = run_onnx(path, ex['input'])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
    return {'right': right, 'wrong': wrong, 'total': right + wrong, 'accuracy': right / (right + wrong) if right + wrong else None}

def model_stats(path):
    model = onnx.load(str(path))
    params = sum(math.prod(init.dims) if init.dims else 1 for init in model.graph.initializer)
    ops = Counter(node.op_type for node in model.graph.node)
    static_mem = 0
    inferred = onnx.shape_inference.infer_shapes(model)
    for v in list(inferred.graph.value_info):
        dims = []
        for d in v.type.tensor_type.shape.dim:
            if not d.HasField('dim_value') or d.dim_value <= 0:
                dims = []; break
            dims.append(d.dim_value)
        if dims:
            static_mem += math.prod(dims) * 4
    return {'params': int(params), 'nodes': len(model.graph.node), 'op_counts': dict(ops), 'file_size_bytes': Path(path).stat().st_size, 'static_memory_bytes': int(static_mem), 'estimated_cost_static': int(params + static_mem)}

def create_zip(model_dir):
    zpath = Path(model_dir) / 'submission.zip'
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(Path(model_dir).glob('task*.onnx')):
            zf.write(p, p.name)
    return zpath

DATA_DIR, OUT_DIR = default_paths()
print('DATA_DIR =', DATA_DIR)
print('OUT_DIR =', OUT_DIR)
print('MODEL_VERSION =', MODEL_VERSION)
print('torch/onnx/ort =', torch.__version__, onnx.__version__, ort.__version__)

task_map = load_task_type_map()
family_df = task_map[task_map.primary_family == FAMILY].copy()
local_df = family_df[family_df.candidate_flags.str.contains('local_3x3_consistent', na=False)].copy()
task_ids = local_df['task_id'].tolist()
print('family tasks:', len(family_df), 'selected local_3x3 tasks:', len(task_ids))
print(local_df[['task_id','n_train','n_test','n_arc_gen','local_3x3_score','local_3x3_conflicts']])
assert len(task_ids) == 8
assert (local_df['local_3x3_score'].astype(float) == 1.0).all()
assert (local_df['local_3x3_conflicts'].astype(int) == 0).all()

for p in OUT_DIR.glob('task*.onnx'):
    p.unlink()

rows = []
for task_id in task_ids:
    print('\n' + '='*80)
    print('training', task_id)
    task = load_task(DATA_DIR, task_id)
    model, info = train_fast(task, task_id)
    row = {'task_id': task_id, **info}
    if model is not None and info.get('ok'):
        path = OUT_DIR / f'{task_id}.onnx'
        export_onnx(model, path)
        val = validate_onnx(path, task)
        row.update({'saved': val['wrong'] == 0, 'path': str(path), 'onnx_visible_right': val['right'], 'onnx_visible_wrong': val['wrong'], 'onnx_visible_total': val['total']})
        if val['wrong'] != 0:
            row['reason'] = 'unique patch exact, but exported ONNX failed full-grid validation'
            path.unlink(missing_ok=True)
    else:
        row['saved'] = False
    rows.append(row)
    print({k: row.get(k) for k in ['task_id','saved','unique_patches','visible_patch_wrong','onnx_visible_wrong','reason']})

result_df = pd.DataFrame(rows)
print(result_df)
saved_count = int(result_df['saved'].sum()) if len(result_df) else 0
print('models saved:', saved_count, '/', len(task_ids))
assert saved_count == len(task_ids), f'Fast H=4 only saved {saved_count}/{len(task_ids)} models; not safe to submit.'
assert (result_df['onnx_visible_wrong'].fillna(1).astype(int) == 0).all()

validate_rows, profile_rows = [], []
for row in rows:
    task = load_task(DATA_DIR, row['task_id'])
    val = validate_onnx(row['path'], task)
    validate_rows.append({'task_id': row['task_id'], **val})
    profile_rows.append({'task_id': row['task_id'], 'model_version': MODEL_VERSION, **model_stats(row['path']), 'visible_accuracy': val['accuracy']})
validate_df = pd.DataFrame(validate_rows)
profile_df = pd.DataFrame(profile_rows)
print(validate_df)
print(profile_df)
print(profile_df[['params','static_memory_bytes','estimated_cost_static']].describe())
assert (validate_df['wrong'].astype(int) == 0).all()
assert (profile_df['visible_accuracy'] == 1.0).all()

zip_path = create_zip(OUT_DIR)
submission_zip = Path('/kaggle/working/submission.zip') if Path('/kaggle/working').exists() else Path.cwd() / 'submission.zip'
shutil.copy2(zip_path, submission_zip)
profile_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_profile.csv'
profile_df.to_csv(profile_path, index=False)
manifest = {'family': FAMILY, 'model_version': MODEL_VERSION, 'hidden_channels': HIDDEN, 'task_count': len(task_ids), 'saved_count': saved_count, 'submission_zip': str(submission_zip), 'profile': str(profile_path)}
with open(OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)
print('submission zip:', submission_zip)
print(manifest)
